# Coding Exercise 5
## Regression Regularisation · Decision Trees · Random Forest · Neural Networks · Spam Detection


## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.linear_model   import LinearRegression, Ridge, Lasso
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.tree            import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble        import RandomForestRegressor
from sklearn.neural_network  import MLPRegressor, MLPClassifier
from sklearn.metrics         import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)


---
# Q1 – OLS, Ridge Regression, Lasso & 5-Fold Cross-Validation

## (a) Data Generation

- $X_k \stackrel{iid}{\sim} \mathcal{N}(\mathbf{0}, I_{20})$, $k=1\ldots100$
- Four random feature indices $i_1,i_2,i_3,i_4 \in \{1,\ldots,20\}$ (without replacement)
- Coefficients $a,b,c,d \sim \mathcal{N}(0,\,0.25)$
- Response: $Y_k = aX_{ki_1} + bX_{ki_2} + cX_{ki_3} + dX_{ki_4} + n_k$, $n_k\sim\mathcal{N}(0,0.01)$


In [ ]:
# ── Data generation ───────────────────────────────────────────────────────────
n_samples = 100
n_features = 20

X_reg = np.random.multivariate_normal(np.zeros(n_features), np.eye(n_features), n_samples)

# Four random feature indices (0-based internally, 1-based for display)
idx4  = np.random.choice(n_features, 4, replace=False)
i1, i2, i3, i4 = idx4
print(f"Active feature indices (1-based): i1={i1+1}, i2={i2+1}, i3={i3+1}, i4={i4+1}")

# Coefficients ~ N(0, 0.25)  [std = 0.5]
a, b, c, d = np.random.normal(0, 0.5, 4)
print(f"True coefficients: a={a:.4f}, b={b:.4f}, c={c:.4f}, d={d:.4f}")

noise = np.random.normal(0, 0.1, n_samples)   # N(0, 0.01) → std=0.1
y_reg = (a*X_reg[:, i1] + b*X_reg[:, i2]
       + c*X_reg[:, i3] + d*X_reg[:, i4] + noise)

print(f"\ny_reg: mean={y_reg.mean():.4f}, std={y_reg.std():.4f}")


## (b) OLS – Closed-Form vs sklearn

Closed form:
$$\hat{\boldsymbol{\beta}}_{\text{OLS}} = (X^\top X)^{-1} X^\top \mathbf{y}$$


In [ ]:
# ── OLS closed form (no bias term – X has zero mean) ─────────────────────────
beta_ols_cf = np.linalg.lstsq(X_reg, y_reg, rcond=None)[0]

# sklearn verification
lr = LinearRegression(fit_intercept=False).fit(X_reg, y_reg)
beta_ols_sk = lr.coef_

print("OLS coefficients (closed-form vs sklearn):")
print(f"  Max absolute difference: {np.max(np.abs(beta_ols_cf - beta_ols_sk)):.2e}")
print(f"  Match: {np.allclose(beta_ols_cf, beta_ols_sk, atol=1e-8)}")

# Display all 20 coefficients
df_ols = pd.DataFrame({
    'Feature':    [f'x{j+1}' for j in range(n_features)],
    'True coef':  [a if j==i1 else b if j==i2 else c if j==i3 else d if j==i4 else 0
                   for j in range(n_features)],
    'OLS (CF)':   beta_ols_cf.round(4),
    'OLS (sk)':   beta_ols_sk.round(4),
})
print("\n", df_ols.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(range(n_features), beta_ols_cf, alpha=0.7, label='OLS (closed form)')
for feat in [i1, i2, i3, i4]:
    ax.axvline(feat, color='r', lw=1.5, ls='--', alpha=0.6)
ax.set_xticks(range(n_features))
ax.set_xticklabels([f'x{j+1}' for j in range(n_features)], fontsize=7)
ax.set_title('OLS Coefficients  (red dashed = active features)')
ax.legend(); plt.tight_layout(); plt.show()


## (c) Ridge Regression – Closed Form vs sklearn

$$\hat{\boldsymbol{\beta}}_{\text{ridge}} = (X^\top X + \alpha I)^{-1} X^\top \mathbf{y}$$


In [ ]:
alpha_ridge = 20     # change freely

# Closed form
beta_ridge_cf = np.linalg.solve(
    X_reg.T @ X_reg + alpha_ridge * np.eye(n_features),
    X_reg.T @ y_reg
)

# sklearn verification
ridge_sk = Ridge(alpha=alpha_ridge, fit_intercept=False).fit(X_reg, y_reg)
beta_ridge_sk = ridge_sk.coef_

print(f"Ridge (alpha={alpha_ridge}) closed-form vs sklearn:")
print(f"  Max absolute difference: {np.max(np.abs(beta_ridge_cf - beta_ridge_sk)):.2e}")
print(f"  Match: {np.allclose(beta_ridge_cf, beta_ridge_sk, atol=1e-6)}")

# Top-5 coefficients by absolute value
abs_order   = np.argsort(np.abs(beta_ridge_cf))[::-1]
top5_idx    = abs_order[:5]
print(f"\nTop-5 feature indices (1-based): {[i+1 for i in top5_idx]}")
print(f"Their values: {beta_ridge_cf[top5_idx].round(4)}")

# Plot
fig, ax = plt.subplots(figsize=(11, 4))
colors = ['steelblue' if j not in top5_idx else 'darkorange' for j in range(n_features)]
ax.bar(range(n_features), np.abs(beta_ridge_cf), color=colors, alpha=0.8)
for feat in [i1, i2, i3, i4]:
    ax.axvline(feat, color='r', lw=1.5, ls='--', alpha=0.6)
ax.set_xticks(range(n_features))
ax.set_xticklabels([f'x{j+1}' for j in range(n_features)], fontsize=7)
ax.set_title(f'Ridge |coefficients| (alpha={alpha_ridge}) — orange=top5, red-dashed=active')
plt.tight_layout(); plt.show()


## (d) Lasso Regression

Lasso does not have a closed-form solution; it uses coordinate descent.  
We use `sklearn.linear_model.Lasso` and sort by absolute value of coefficients.


In [ ]:
alpha_lasso = 0.01    # much smaller alpha for Lasso (different scale than Ridge)

lasso_sk = Lasso(alpha=alpha_lasso, fit_intercept=False, max_iter=10000).fit(X_reg, y_reg)
beta_lasso = lasso_sk.coef_

abs_order_lasso = np.argsort(np.abs(beta_lasso))[::-1]
top5_lasso      = abs_order_lasso[:5]

print(f"Lasso (alpha={alpha_lasso})")
print(f"  Non-zero coefficients: {np.sum(beta_lasso != 0)}/{n_features}")
print(f"  Top-5 indices (1-based): {[i+1 for i in top5_lasso]}")
print(f"  Top-5 values : {beta_lasso[top5_lasso].round(4)}")

# Compare Ridge vs Lasso
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, beta, title, top5, sk_alpha in [
        (axes[0], beta_ridge_cf, f'Ridge (α={alpha_ridge})', top5_idx, alpha_ridge),
        (axes[1], beta_lasso,    f'Lasso (α={alpha_lasso})', top5_lasso, alpha_lasso)]:
    colors = ['darkorange' if j in top5 else 'steelblue' for j in range(n_features)]
    ax.bar(range(n_features), beta, color=colors, alpha=0.8)
    for feat in [i1, i2, i3, i4]:
        ax.axvline(feat, color='r', lw=1.5, ls='--', alpha=0.6)
    ax.set_xticks(range(n_features))
    ax.set_xticklabels([f'x{j+1}' for j in range(n_features)], fontsize=7)
    ax.set_title(title + '  — orange=top5, red-dashed=active')
plt.tight_layout(); plt.show()


## (e) 5-Fold Cross-Validation for Best α – From Scratch + GridSearchCV

We search over:
$$\alpha \in \{1, 2, 5, 10, 20, 50, 100, 1000, 10000\}$$

**From scratch:** split data into 5 folds manually, compute validation MSE  
for each (model, α) combination, then pick the α with lowest mean CV error.  
**sklearn GridSearchCV:** used to verify.


In [ ]:
alphas = [1, 2, 5, 10, 20, 50, 100, 1000, 10000]
K_FOLDS = 5

def kfold_cv_manual(X, y, model_fn, alphas, k=K_FOLDS):
    """
    Manual K-fold CV.
    model_fn(X_tr, y_tr, X_val, alpha) -> val predictions
    Returns: (alphas, mean_cv_errors, std_cv_errors)
    """
    n = len(y)
    idx   = np.random.permutation(n)
    folds = np.array_split(idx, k)

    cv_errors = np.zeros((len(alphas), k))
    for a_i, alpha in enumerate(alphas):
        for f_i in range(k):
            val_idx  = folds[f_i]
            tr_idx   = np.concatenate([folds[j] for j in range(k) if j != f_i])
            X_tr, y_tr   = X[tr_idx],  y[tr_idx]
            X_val, y_val = X[val_idx], y[val_idx]
            y_pred = model_fn(X_tr, y_tr, X_val, alpha)
            cv_errors[a_i, f_i] = np.mean((y_val - y_pred) ** 2)

    return np.array(alphas), cv_errors.mean(axis=1), cv_errors.std(axis=1)


# ── Ridge CV ─────────────────────────────────────────────────────────────────
def ridge_predict(X_tr, y_tr, X_val, alpha):
    w = np.linalg.solve(X_tr.T @ X_tr + alpha * np.eye(X_tr.shape[1]), X_tr.T @ y_tr)
    return X_val @ w

alphas_r, cv_mean_r, cv_std_r = kfold_cv_manual(X_reg, y_reg, ridge_predict, alphas)
best_alpha_ridge_cv = alphas_r[np.argmin(cv_mean_r)]
print(f"Ridge 5-fold CV best α (manual) : {best_alpha_ridge_cv}")

# sklearn GridSearchCV for Ridge
ridge_gs = GridSearchCV(
    Ridge(fit_intercept=False),
    param_grid={'alpha': alphas},
    cv=K_FOLDS, scoring='neg_mean_squared_error'
).fit(X_reg, y_reg)
print(f"Ridge 5-fold CV best α (sklearn): {ridge_gs.best_params_['alpha']}")

# ── Lasso CV ──────────────────────────────────────────────────────────────────
def lasso_predict(X_tr, y_tr, X_val, alpha):
    m = Lasso(alpha=alpha, fit_intercept=False, max_iter=10000).fit(X_tr, y_tr)
    return X_val @ m.coef_

alphas_l, cv_mean_l, cv_std_l = kfold_cv_manual(X_reg, y_reg, lasso_predict, alphas)
best_alpha_lasso_cv = alphas_l[np.argmin(cv_mean_l)]
print(f"Lasso 5-fold CV best α (manual) : {best_alpha_lasso_cv}")

lasso_gs = GridSearchCV(
    Lasso(fit_intercept=False, max_iter=10000),
    param_grid={'alpha': alphas},
    cv=K_FOLDS, scoring='neg_mean_squared_error'
).fit(X_reg, y_reg)
print(f"Lasso 5-fold CV best α (sklearn): {lasso_gs.best_params_['alpha']}")

# ── Plot CV curves ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, means, stds, best, title in [
        (axes[0], cv_mean_r, cv_std_r, best_alpha_ridge_cv, 'Ridge'),
        (axes[1], cv_mean_l, cv_std_l, best_alpha_lasso_cv, 'Lasso')]:
    ax.errorbar(range(len(alphas)), means, yerr=stds,
                fmt='b-o', ms=5, capsize=4, label='CV MSE ± 1 std')
    best_i = alphas.index(best)
    ax.axvline(best_i, color='r', ls='--', label=f'best α={best}')
    ax.set_xticks(range(len(alphas)))
    ax.set_xticklabels(alphas, rotation=45, fontsize=8)
    ax.set_xlabel('α'); ax.set_ylabel('CV MSE')
    ax.set_title(f'{title} 5-Fold CV'); ax.legend()
plt.tight_layout(); plt.show()

print(f"\nSummary:")
print(f"  Ridge best α: manual={best_alpha_ridge_cv},  GridSearchCV={ridge_gs.best_params_['alpha']}")
print(f"  Lasso best α: manual={best_alpha_lasso_cv},  GridSearchCV={lasso_gs.best_params_['alpha']}")


---
# Q2 – Classification Decision Tree (from Scratch + sklearn)

## (a) Dataset (same as Exercise 4, Q1)


In [ ]:
# ── Reproduce Exercise 4 dataset ─────────────────────────────────────────────
np.random.seed(42)
N_MEANS, N_PER_MEAN = 10, 10
COV_SMALL = 0.1 * np.eye(2)

m_pos = np.random.multivariate_normal([1, 0], np.eye(2), N_MEANS)
m_neg = np.random.multivariate_normal([0, 1], np.eye(2), N_MEANS)

def sample_from_means(means, n_per, cov=COV_SMALL):
    return np.vstack([np.random.multivariate_normal(mu, cov, n_per) for mu in means])

X_cls = np.vstack([sample_from_means(m_pos, N_PER_MEAN),
                   sample_from_means(m_neg, N_PER_MEAN)])   # (200, 2)
y_cls = np.array([1]*100 + [-1]*100)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X_cls[y_cls== 1,0], X_cls[y_cls== 1,1], c='steelblue', s=25,
           alpha=0.7, label='y=+1', edgecolors='k', lw=0.3)
ax.scatter(X_cls[y_cls==-1,0], X_cls[y_cls==-1,1], c='tomato',    s=25,
           alpha=0.7, label='y=-1', edgecolors='k', lw=0.3)
ax.set_title('Dataset (200 points)'); ax.legend(); plt.tight_layout(); plt.show()


## (b) & (c) Binary Classification Tree from Scratch – Gini Index

### Gini Index
$$\text{Gini}(S) = 1 - \sum_c p_c^2$$

At each node we search over all features and all split thresholds to minimise the
weighted Gini impurity of the two child nodes.

**Stopping rules:**
- Node is pure (misclassification = 0)
- Node has ≤ 10 points
- Maximum depth = 3


In [ ]:
# ── Gini-based Decision Tree from scratch ────────────────────────────────────

def gini_impurity(y):
    """Gini impurity of a label array."""
    if len(y) == 0:
        return 0.0
    classes, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return 1.0 - np.sum(p ** 2)


def best_split(X, y):
    """
    Find the (feature, threshold) pair that minimises weighted Gini impurity.
    Returns (best_feature_idx, best_threshold, best_weighted_gini).
    """
    n, n_feat = X.shape
    best_gini = np.inf
    best_feat, best_thr = None, None

    for feat in range(n_feat):
        # Candidate thresholds: midpoints between consecutive sorted unique values
        vals  = np.unique(X[:, feat])
        thresholds = (vals[:-1] + vals[1:]) / 2

        for thr in thresholds:
            left  = y[X[:, feat] <= thr]
            right = y[X[:, feat] >  thr]
            if len(left) == 0 or len(right) == 0:
                continue
            weighted = (len(left)  * gini_impurity(left)
                      + len(right) * gini_impurity(right)) / n
            if weighted < best_gini:
                best_gini = weighted
                best_feat, best_thr = feat, thr

    return best_feat, best_thr, best_gini


def majority_class(y):
    vals, counts = np.unique(y, return_counts=True)
    return vals[np.argmax(counts)]


def build_tree(X, y, depth=0, max_depth=3, min_samples=10):
    """
    Recursively build a classification tree using Gini impurity.
    Returns a dict representing a node.
    """
    node = {
        'n_samples'  : len(y),
        'gini'       : gini_impurity(y),
        'prediction' : majority_class(y),
        'depth'      : depth,
    }

    # Stopping conditions
    misclassified = np.sum(y != node['prediction'])
    if misclassified == 0 or len(y) <= min_samples or depth >= max_depth:
        node['is_leaf'] = True
        return node

    feat, thr, _ = best_split(X, y)
    if feat is None:                   # all points identical
        node['is_leaf'] = True
        return node

    left_mask  = X[:, feat] <= thr
    right_mask = ~left_mask

    node['is_leaf']    = False
    node['feat']       = feat
    node['threshold']  = thr
    node['left']       = build_tree(X[left_mask],  y[left_mask],  depth+1, max_depth, min_samples)
    node['right']      = build_tree(X[right_mask], y[right_mask], depth+1, max_depth, min_samples)
    return node


def predict_tree(node, X):
    """Predict labels for all rows of X using the fitted tree."""
    preds = np.empty(len(X), dtype=int)
    for i, x in enumerate(X):
        n = node
        while not n['is_leaf']:
            n = n['left'] if x[n['feat']] <= n['threshold'] else n['right']
        preds[i] = n['prediction']
    return preds


def print_tree(node, feature_names=None, indent=''):
    """Pretty-print the tree structure."""
    if node['is_leaf']:
        print(f"{indent}LEAF → predict={node['prediction']}  "
              f"n={node['n_samples']}  gini={node['gini']:.4f}")
    else:
        fname = feature_names[node['feat']] if feature_names else f"x{node['feat']}"
        print(f"{indent}[{fname} <= {node['threshold']:.4f}]  "
              f"n={node['n_samples']}  gini={node['gini']:.4f}")
        print(f"{indent}├── LEFT:")
        print_tree(node['left'],  feature_names, indent + '│   ')
        print(f"{indent}└── RIGHT:")
        print_tree(node['right'], feature_names, indent + '    ')


# ── Build tree ────────────────────────────────────────────────────────────────
MAX_DEPTH   = 3
MIN_SAMPLES = 10

tree_scratch = build_tree(X_cls, y_cls, max_depth=MAX_DEPTH, min_samples=MIN_SAMPLES)
print("=== Tree structure (from scratch) ===")
print_tree(tree_scratch, feature_names=['x1', 'x2'])

y_pred_scratch = predict_tree(tree_scratch, X_cls)
train_err_tree = np.mean(y_pred_scratch != y_cls)
print(f"\nTraining error (scratch tree): {train_err_tree:.4f}  ({int(train_err_tree*200)}/200)")


## (c) sklearn DecisionTreeClassifier – Verification

In [ ]:
# ── sklearn tree ──────────────────────────────────────────────────────────────
clf_sk = DecisionTreeClassifier(
    criterion='gini',
    max_depth=MAX_DEPTH,
    min_samples_leaf=MIN_SAMPLES,
    random_state=42
).fit(X_cls, y_cls)

y_pred_sk  = clf_sk.predict(X_cls)
err_sk     = np.mean(y_pred_sk != y_cls)
print(f"Training error (sklearn):        {err_sk:.4f}  ({int(err_sk*200)}/200)")
print(f"Training error (scratch):        {train_err_tree:.4f}")
print(f"Predictions match: {np.mean(y_pred_scratch == y_pred_sk):.2%}")

# Plot sklearn tree
fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(clf_sk, feature_names=['x1','x2'],
          class_names=['-1','1'], filled=True, ax=ax, fontsize=9)
ax.set_title('sklearn DecisionTreeClassifier (depth≤3)')
plt.tight_layout(); plt.show()


## (d) Classification Rectangles + Scatter Plot

In [ ]:
def plot_decision_regions(predict_fn, X, y, ax=None, grid_res=200, title=''):
    """Colour decision regions by evaluating predict_fn on a dense grid."""
    from matplotlib.colors import ListedColormap
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 6))
    pad = 0.4
    x1_min, x1_max = X[:,0].min()-pad, X[:,0].max()+pad
    x2_min, x2_max = X[:,1].min()-pad, X[:,1].max()+pad
    xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, grid_res),
                           np.linspace(x2_min, x2_max, grid_res))
    grid = np.column_stack([xx1.ravel(), xx2.ravel()])
    Z = predict_fn(grid).reshape(xx1.shape)

    cmap = ListedColormap(['#ffcccc','#cce0ff'])
    ax.contourf(xx1, xx2, Z, levels=[-2, 0, 2], cmap=cmap, alpha=0.45)
    ax.contour( xx1, xx2, Z, levels=[0], colors='k', linewidths=1.2)
    ax.scatter(X[y== 1,0], X[y== 1,1], c='steelblue', s=25,
               alpha=0.8, label='y=+1', edgecolors='k', lw=0.3)
    ax.scatter(X[y==-1,0], X[y==-1,1], c='tomato',    s=25,
               alpha=0.8, label='y=-1', edgecolors='k', lw=0.3)
    ax.set_xlabel('x1'); ax.set_ylabel('x2')
    ax.legend(fontsize=9); ax.set_title(title)
    return ax


fig, axes = plt.subplots(1, 2, figsize=(13, 6))
plot_decision_regions(
    lambda Xg: predict_tree(tree_scratch, Xg),
    X_cls, y_cls, ax=axes[0],
    title=f'Scratch Tree (depth≤{MAX_DEPTH})  train err={train_err_tree:.3f}')
plot_decision_regions(
    lambda Xg: clf_sk.predict(Xg),
    X_cls, y_cls, ax=axes[1],
    title=f'sklearn Tree (depth≤{MAX_DEPTH})  train err={err_sk:.3f}')
plt.tight_layout(); plt.show()

print(f"Training error (scratch): {train_err_tree:.4f}")
print(f"Training error (sklearn): {err_sk:.4f}")


---
# Q3 – Carseats: Regression Tree & Random Forest

## (a) Load & Preprocess

Encoding: `Good→1`, `Medium→0`, `Bad→-1`, `Yes→1`, `No→0`


In [ ]:
# ── Load Carseats (update path if needed) ────────────────────────────────────
import os
_cs_paths = ['Carseats.csv', '/home/claude/Carseats.csv',
             '/mnt/user-data/uploads/Carseats.csv']
cs_path = next((p for p in _cs_paths if os.path.exists(p)), None)
assert cs_path, "Carseats.csv not found – place it in the working directory."

df_cs = pd.read_csv(cs_path)
print(f"Loaded: {df_cs.shape}  columns: {list(df_cs.columns)}")
print(df_cs.head(3))

# ── Encode categorical columns ────────────────────────────────────────────────
encode_map = {'Good': 1, 'Medium': 0, 'Bad': -1, 'Yes': 1, 'No': 0}
df_cs = df_cs.replace(encode_map)
print("\nAfter encoding:")
print(df_cs.dtypes)
print(df_cs.head(3))

# ── Train / test split (80 / 20) ─────────────────────────────────────────────
np.random.seed(42)
n_cs   = len(df_cs)
idx_cs = np.random.permutation(n_cs)
n_tr   = int(0.8 * n_cs)
tr_idx, te_idx = idx_cs[:n_tr], idx_cs[n_tr:]

X_cs = df_cs.drop(columns='Sales').values.astype(float)
y_cs = df_cs['Sales'].values.astype(float)
feat_names_cs = list(df_cs.drop(columns='Sales').columns)

X_cs_tr, y_cs_tr = X_cs[tr_idx], y_cs[tr_idx]
X_cs_te, y_cs_te = X_cs[te_idx], y_cs[te_idx]
print(f"\nTrain: {X_cs_tr.shape[0]}  Test: {X_cs_te.shape[0]}")


## (b) DecisionTreeRegressor

In [ ]:
dt_reg = DecisionTreeRegressor(
    min_samples_leaf=10,
    random_state=42
).fit(X_cs_tr, y_cs_tr)

y_pred_cs_te = dt_reg.predict(X_cs_te)
test_mse_dt  = mean_squared_error(y_cs_te, y_pred_cs_te)
test_rmse_dt = np.sqrt(test_mse_dt)

print(f"Decision Tree Regressor:")
print(f"  Tree depth : {dt_reg.get_depth()}")
print(f"  Test MSE   : {test_mse_dt:.4f}")
print(f"  Test RMSE  : {test_rmse_dt:.4f}")

# Plot tree (truncated for readability)
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(dt_reg, feature_names=feat_names_cs, filled=True,
          max_depth=3, ax=ax, fontsize=7, rounded=True)
ax.set_title('DecisionTreeRegressor (first 3 levels shown)')
plt.tight_layout(); plt.show()

# Residual plot
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_cs_te, y_pred_cs_te, alpha=0.6, s=20)
lim = [min(y_cs_te.min(), y_pred_cs_te.min())-0.5,
       max(y_cs_te.max(), y_pred_cs_te.max())+0.5]
ax.plot(lim, lim, 'r--', lw=1.5, label='perfect prediction')
ax.set_xlabel('True Sales'); ax.set_ylabel('Predicted Sales')
ax.set_title(f'DT Regressor  Test RMSE={test_rmse_dt:.3f}')
ax.legend(); plt.tight_layout(); plt.show()


## (c) Random Forest (B=100, max_features=4)

In [ ]:
rf_reg = RandomForestRegressor(
    n_estimators=100,
    max_features=4,      # 4 of 10 predictors at each split
    min_samples_leaf=10,
    random_state=42
).fit(X_cs_tr, y_cs_tr)

y_pred_rf_te  = rf_reg.predict(X_cs_te)
test_mse_rf   = mean_squared_error(y_cs_te, y_pred_rf_te)
test_rmse_rf  = np.sqrt(test_mse_rf)

print(f"Random Forest (B=100, max_features=4):")
print(f"  Test MSE  : {test_mse_rf:.4f}")
print(f"  Test RMSE : {test_rmse_rf:.4f}")
print(f"  Improvement over single DT: {(test_rmse_dt - test_rmse_rf)/test_rmse_dt*100:.1f}%")

# Feature importance
importances = rf_reg.feature_importances_
order = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(feat_names_cs)),
       importances[order], color='steelblue', alpha=0.8)
ax.set_xticks(range(len(feat_names_cs)))
ax.set_xticklabels([feat_names_cs[i] for i in order], rotation=45, ha='right')
ax.set_title(f'RF Feature Importances  (Test RMSE={test_rmse_rf:.3f})')
plt.tight_layout(); plt.show()

print(f"\nModel comparison:")
print(f"  {'Model':<30} {'Test RMSE':>10}")
print(f"  {'DecisionTreeRegressor':<30} {test_rmse_dt:>10.4f}")
print(f"  {'RandomForestRegressor':<30} {test_rmse_rf:>10.4f}")


---
# Q4 – Neural Network for $y = \|\mathbf{x}\|_2^2$

## (a) Data Generation


In [ ]:
np.random.seed(42)
N_NN = 10_000

X_nn_tr = np.random.uniform(-10, 10, (N_NN, 2))
y_nn_tr = np.sum(X_nn_tr**2, axis=1)          # y = x1^2 + x2^2

print(f"Training data: X{X_nn_tr.shape}, y range [{y_nn_tr.min():.1f}, {y_nn_tr.max():.1f}]")


## (b) 5-Fold CV to Select M and Activation Function


In [ ]:
M_values   = [5, 10, 20, 50, 100]
activations = ['logistic', 'tanh', 'relu']
K_NN = 5

kf = KFold(n_splits=K_NN, shuffle=True, random_state=0)

cv_results = {}
print("Running 5-fold CV (this may take ~1 min) ...")
for act in activations:
    for M in M_values:
        fold_errs = []
        for tr_idx, val_idx in kf.split(X_nn_tr):
            Xtr, ytr = X_nn_tr[tr_idx], y_nn_tr[tr_idx]
            Xval, yval = X_nn_tr[val_idx], y_nn_tr[val_idx]
            mlp = MLPRegressor(
                hidden_layer_sizes=(M,),
                activation=act,
                max_iter=500,
                random_state=42,
                early_stopping=True,
                validation_fraction=0.1
            ).fit(Xtr, ytr)
            fold_errs.append(mean_squared_error(yval, mlp.predict(Xval)))
        cv_results[(act, M)] = np.mean(fold_errs)
        print(f"  {act:8s}  M={M:3d}  CV-MSE={cv_results[(act,M)]:.4f}")

best_key   = min(cv_results, key=cv_results.get)
best_act, best_M = best_key
print(f"\nBest: activation='{best_act}', M={best_M}, CV-MSE={cv_results[best_key]:.4f}")

# ── Plot CV heatmap ───────────────────────────────────────────────────────────
cv_mat = np.array([[cv_results[(a, M)] for M in M_values] for a in activations])
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(cv_mat, aspect='auto', cmap='YlOrRd_r')
ax.set_xticks(range(len(M_values))); ax.set_xticklabels(M_values)
ax.set_yticks(range(len(activations))); ax.set_yticklabels(activations)
ax.set_xlabel('M (hidden nodes)'); ax.set_ylabel('Activation')
ax.set_title('5-Fold CV MSE Heatmap')
plt.colorbar(im, ax=ax)
for i in range(len(activations)):
    for j in range(len(M_values)):
        ax.text(j, i, f'{cv_mat[i,j]:.1f}', ha='center', va='center', fontsize=8)
plt.tight_layout(); plt.show()

# ── Fit best model on full training data ──────────────────────────────────────
best_mlp = MLPRegressor(
    hidden_layer_sizes=(best_M,),
    activation=best_act,
    max_iter=1000,
    random_state=42
).fit(X_nn_tr, y_nn_tr)

y_pred_tr_nn = best_mlp.predict(X_nn_tr)
train_mse_nn = mean_squared_error(y_nn_tr, y_pred_tr_nn)
print(f"\nFinal model  activation='{best_act}', M={best_M}")
print(f"  Training MSE  : {train_mse_nn:.4f}")
print(f"  Training RMSE : {np.sqrt(train_mse_nn):.4f}")


## (c) Test Error & 3D Comparison Plot

In [ ]:
# ── Test data ─────────────────────────────────────────────────────────────────
X_nn_te = np.random.uniform(-10, 10, (N_NN, 2))
y_nn_te = np.sum(X_nn_te**2, axis=1)

y_pred_te_nn = best_mlp.predict(X_nn_te)
test_mse_nn  = mean_squared_error(y_nn_te, y_pred_te_nn)
print(f"Test MSE  : {test_mse_nn:.4f}")
print(f"Test RMSE : {np.sqrt(test_mse_nn):.4f}")

# ── 3D plot: true vs NN ───────────────────────────────────────────────────────
grid_pts = 60
x1g = np.linspace(-5, 5, grid_pts)
x2g = np.linspace(-5, 5, grid_pts)
Xg1, Xg2 = np.meshgrid(x1g, x2g)
Xgrid     = np.column_stack([Xg1.ravel(), Xg2.ravel()])

Z_true = (Xg1**2 + Xg2**2)
Z_nn   = best_mlp.predict(Xgrid).reshape(Xg1.shape)

fig = plt.figure(figsize=(14, 6))

ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(Xg1, Xg2, Z_true, cmap='viridis', alpha=0.85)
ax1.set_title('True: $g(x,y) = x^2 + y^2$')
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('z')

ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_surface(Xg1, Xg2, Z_nn, cmap='plasma', alpha=0.85)
ax2.set_title(f'NN Prediction  (M={best_M}, {best_act})
Test RMSE={np.sqrt(test_mse_nn):.3f}')
ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.set_zlabel('z')

plt.tight_layout(); plt.show()


---
# Q5 – Spam Email Detection (sklearn MLPClassifier)

> **Data:** Download from https://archive.ics.uci.edu/dataset/94/spambase  
> Place `spambase.csv` (features + `spam` label column) in the working directory.  
> If unavailable, the code below falls back to a synthetic dataset with the same schema.

## (a) sklearn MLPClassifier


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics       import accuracy_score, classification_report, confusion_matrix

# ── Load data ─────────────────────────────────────────────────────────────────
import os
_sp_paths = ['spambase.csv', '/home/claude/spambase.csv',
             '/mnt/user-data/uploads/spambase.csv']
sp_path   = next((p for p in _sp_paths if os.path.exists(p)), None)

if sp_path:
    df_sp = pd.read_csv(sp_path)
    print(f"Loaded spambase: {df_sp.shape}")
    # Last column is the label (spam=1/0)
    label_col = df_sp.columns[-1]
    X_sp = df_sp.drop(columns=label_col).values.astype(float)
    y_sp = df_sp[label_col].values.astype(int)
else:
    raise FileNotFoundError(
        "spambase.csv not found. Download from "
        "https://archive.ics.uci.edu/dataset/94/spambase "
        "and place in the working directory.")

print(f"Features: {X_sp.shape[1]}, Samples: {X_sp.shape[0]}")
print(f"Spam rate: {y_sp.mean():.3f}")

# ── Train / test split (75% / 25%) ───────────────────────────────────────────
np.random.seed(42)
n_sp   = len(y_sp)
idx_sp = np.random.permutation(n_sp)
n_tr_sp = int(0.75 * n_sp)
tr_sp, te_sp = idx_sp[:n_tr_sp], idx_sp[n_tr_sp:]

X_sp_tr, y_sp_tr = X_sp[tr_sp], y_sp[tr_sp]
X_sp_te, y_sp_te = X_sp[te_sp], y_sp[te_sp]

# ── Standardise (important for MLP) ──────────────────────────────────────────
scaler = StandardScaler().fit(X_sp_tr)
X_sp_tr_sc = scaler.transform(X_sp_tr)
X_sp_te_sc = scaler.transform(X_sp_te)

# ── 5-fold CV to choose M and activation ─────────────────────────────────────
from sklearn.model_selection import cross_val_score

M_sp   = [50, 100]          # reduce search for speed
act_sp = ['relu', 'tanh']
kf_sp  = KFold(n_splits=5, shuffle=True, random_state=0)

best_cv_sp, best_cfg_sp = -np.inf, None
print("CV search (sklearn MLP):")
for act in act_sp:
    for M in M_sp:
        mlp_cv = MLPClassifier(hidden_layer_sizes=(M,), activation=act,
                               max_iter=500, random_state=42)
        scores = cross_val_score(mlp_cv, X_sp_tr_sc, y_sp_tr,
                                 cv=kf_sp, scoring='accuracy')
        print(f"  {act:6s}  M={M:3d}  CV acc={scores.mean():.4f} ± {scores.std():.4f}")
        if scores.mean() > best_cv_sp:
            best_cv_sp, best_cfg_sp = scores.mean(), (act, M)

best_act_sp, best_M_sp = best_cfg_sp
print(f"\nBest config: activation='{best_act_sp}', M={best_M_sp}")

# ── Final fit ─────────────────────────────────────────────────────────────────
mlp_spam = MLPClassifier(
    hidden_layer_sizes=(best_M_sp,),
    activation=best_act_sp,
    max_iter=1000,
    random_state=42
).fit(X_sp_tr_sc, y_sp_tr)

y_tr_pred_sp = mlp_spam.predict(X_sp_tr_sc)
y_te_pred_sp = mlp_spam.predict(X_sp_te_sc)

train_err_sp = 1 - accuracy_score(y_sp_tr, y_tr_pred_sp)
test_err_sp  = 1 - accuracy_score(y_sp_te, y_te_pred_sp)

print(f"\n=== sklearn MLPClassifier  (M={best_M_sp}, activation='{best_act_sp}') ===")
print(f"  Training error : {train_err_sp:.4f}  ({train_err_sp*100:.2f}%)")
print(f"  Test error     : {test_err_sp:.4f}  ({test_err_sp*100:.2f}%)")
print(f"\nClassification report (test set):")
print(classification_report(y_sp_te, y_te_pred_sp, target_names=['not spam','spam']))

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_sp_te, y_te_pred_sp)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['not spam','spam']); ax.set_yticklabels(['not spam','spam'])
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix – sklearn MLP\nTest error={test_err_sp:.4f}')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=14,
                color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.colorbar(im); plt.tight_layout(); plt.show()


---
## Summary

| Q | Task | Method | Key point |
|---|------|--------|-----------|
| 1a | Sparse linear regression | Data gen | 4 active features out of 20 |
| 1b | OLS | Normal equations | Verified vs sklearn |
| 1c | Ridge | Closed form $(X^⊤X+αI)^{-1}X^⊤y$ | Top-5 features identified |
| 1d | Lasso | sklearn (coord. descent) | Sparse solution |
| 1e | 5-fold CV | Manual folds + GridSearchCV | Best α for Ridge & Lasso |
| 2b-c | Decision tree | Gini impurity from scratch | Max depth 3, min 10 pts |
| 2d | Viz | Decision rectangles | Train error printed |
| 3b | Regression tree | DecisionTreeRegressor | Min 10 pts per leaf |
| 3c | Random forest | B=100, 4 features/split | Feature importance plotted |
| 4b | MLP regression | 5-fold CV over M ∈{5,10,20,50,100} & 3 activations | $y=\|x\|^2$ |
| 4c | 3D plot | True vs NN surface | Test error printed |
| 5a | Spam MLP | sklearn, best M & activation | Train & test error |
